# Generate synthetic data 
(Có causal structure)

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 5000

income = np.random.normal(30, 10, n)
tenure = np.random.exponential(3, n)
past_spend = income * 2 + np.random.normal(0, 10, n)
risk_score = np.random.beta(2, 5, n)

logit_p = (
    0.02 * income
    + 0.05 * past_spend
    - 0.8 * risk_score
    + np.random.normal(0, 1.0, n)   # 🔴 noise tạo overlap
)

p_treat = 1 / (1 + np.exp(-logit_p))

T = np.random.binomial(1, p_treat)


# True treatment effect
TRUE_ATE = 20

Y = (
    50
    + 1.5 * past_spend
    - 10 * risk_score
    + TRUE_ATE * T
    + np.random.normal(0, 10, n)
)

df = pd.DataFrame({
    "income": income,
    "tenure": tenure,
    "past_spend": past_spend,
    "risk_score": risk_score,
    "T": T,
    "Y": Y
})


df.head()

,income,tenure,past_spend,risk_score,T,Y
0,34.967142,0.552622,60.549612,0.345068,1,152.218099
1,28.617357,0.631921,51.770638,0.058941,1,146.093072
2,36.476885,1.852518,70.131365,0.146886,1,178.802527
3,45.230299,1.011793,100.968340,0.307310,1,224.282615
4,27.658466,0.852915,43.687543,0.197901,1,141.161449


In [2]:
df.tail()

,income,tenure,past_spend,risk_score,T,Y
4995,29.510350,0.453463,65.972360,0.282537,1,166.928944
4996,37.114106,6.529552,101.587813,0.525369,1,199.570140
4997,61.129102,3.925475,127.945901,0.440318,1,247.443091
4998,38.080362,0.539786,69.861861,0.569600,1,168.495877
4999,21.519344,2.502874,46.517849,0.122387,1,136.587135


# Naive ATE 
Sai vì confounding (vì khách được chọn vốn đã chi tiêu cao hơn)

In [3]:
naive_ate = df[df["T"] == 1]["Y"].mean() - df[df["T"] == 0]["Y"].mean()
naive_ate


54.06911798330863

# Regression Adjustment (Causal Linear Model)
coef(T) ≈ ATE

In [4]:
import statsmodels.api as sm

X = df[["T", "income", "tenure", "past_spend", "risk_score"]]
X = sm.add_constant(X)
model = sm.OLS(df.Y, X).fit()

model.params["T"]


20.172121502950137

# Propensity Score (IPW)

## Ước lượng propensity

In [5]:
from sklearn.linear_model import LogisticRegression

ps_model = LogisticRegression(max_iter=1000)

ps_model.fit(
    df[["income", "tenure", "past_spend", "risk_score"]],
    df["T"]     # ✅ đúng
)

df["ps"] = ps_model.predict_proba(
    df[["income", "tenure", "past_spend", "risk_score"]]
)[:, 1]

df.head()


,income,tenure,past_spend,risk_score,T,Y,ps
0,34.967142,0.552622,60.549612,0.345068,1,152.218099,0.946848
1,28.617357,0.631921,51.770638,0.058941,1,146.093072,0.918673
2,36.476885,1.852518,70.131365,0.146886,1,178.802527,0.966452
3,45.230299,1.011793,100.968340,0.307310,1,224.282615,0.991887
4,27.658466,0.852915,43.687543,0.197901,1,141.161449,0.885863


## IPW ATE

In [6]:
import numpy as np

ate_ipw = (
    (df["T"] * df["Y"] / df["ps"]).mean()
    - ((1 - df["T"]) * df["Y"] / (1 - df["ps"])).mean()
)

ate_ipw


20.082434245114598

# Doubly Robust ATE

In [9]:
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold

##
X = df[["income", "tenure", "past_spend", "risk_score"]].values
T = df["T"].values
Y = df["Y"].values


## Cross-fitted Doubly Robust 
kf = KFold(n_splits=2, shuffle=True, random_state=42)

dr_scores = []

for train_idx, test_idx in kf.split(X):
    X_tr, X_te = X[train_idx], X[test_idx]
    T_tr, T_te = T[train_idx], T[test_idx]
    Y_tr, Y_te = Y[train_idx], Y[test_idx]

    # Propensity
    ps_model = LogisticRegression(max_iter=1000)
    ps_model.fit(X_tr, T_tr)
    ps = ps_model.predict_proba(X_te)[:, 1]
    ps = np.clip(ps, 0.05, 0.95)   # 🔴 clip mạnh

    # Outcome models (LINEAR)
    mu1_model = LinearRegression().fit(X_tr[T_tr == 1], Y_tr[T_tr == 1])
    mu0_model = LinearRegression().fit(X_tr[T_tr == 0], Y_tr[T_tr == 0])

    mu1 = mu1_model.predict(X_te)
    mu0 = mu0_model.predict(X_te)

    dr = (
        mu1 - mu0
        + T_te * (Y_te - mu1) / ps
        - (1 - T_te) * (Y_te - mu0) / (1 - ps)
    )

    dr_scores.append(dr)


ate_dr = np.mean(np.concatenate(dr_scores))
ate_dr


20.688532103585416